# 02 · Market Data Context & Curves

Deterministic market data is the foundation for every valuation. This notebook walks through the concrete objects that populate `MarketContext` — discount curves, rate indices, credit curves, inflation projections, volatility surfaces, FX matrices, and scalar price feeds.

### Learning Objectives
- Build discount, forward, hazard, and inflation curves using `finstack.core.market_data.term_structures`
- Attach volatility surfaces, FX matrices, and scalar prices to a `MarketContext`
- Inspect basic analytics such as zero rates, survival probabilities, and implied vol grids
- Feed the resulting context into the valuations pricer registry to price an instrument and request metrics

In [ ]:
# Imports and shared helpers
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD, EUR, JPY
from finstack.core.market_data import MarketContext
from finstack.core.market_data.fx import FxMatrix
from finstack.core.market_data.scalars import MarketScalar
from finstack.core.market_data.surfaces import VolSurface
from finstack.core.market_data.term_structures import (
    DiscountCurve,
    ForwardCurve,
    HazardCurve,
    InflationCurve,
    BaseCorrelationCurve,
)
from finstack.valuations.instruments import Bond
from finstack.valuations.pricer import create_standard_registry


## 1. Discount Curves
Discount curves anchor discount factors and zero rates used by every deterministic pricer. Provide an ID, base date, and `(time, df)` pairs.

Below we build simple USD OIS and EUR OIS curves and inspect zero rates at multiple maturities.

In [ ]:
as_of = date(2025, 1, 2)

usd_ois = DiscountCurve(
    "USD-OIS",
    as_of,
    [
        (0.0, 1.0),
        (0.5, 0.9972),
        (1.0, 0.9940),
        (3.0, 0.9725),
        (5.0, 0.9440),
        (10.0, 0.8800),
    ],
    interp="monotone_convex",
    extrapolation="flat_zero",
)
eur_ois = DiscountCurve(
    "EUR-OIS",
    as_of,
    [
        (0.0, 1.0),
        (0.5, 0.9985),
        (1.0, 0.9960),
        (3.0, 0.9820),
        (5.0, 0.9575),
        (10.0, 0.9100),
    ],
)

print("USD-OIS points:", usd_ois.points)
for t in (0.5, 2.0, 7.0):
    print(f"  t={t:>4.1f}y zero={usd_ois.zero(t):.4%} df={usd_ois.df(t):.6f}")
print()
print("EUR-OIS zero curve sample:")
for t in (0.25, 1.0, 4.0):
    print(f"  t={t:>4.1f}y zero={eur_ois.zero(t):.4%}")


## 2. Forward Curves (Rate Indices)
Forward curves represent projected index fixings (e.g., SOFR 3M versus 6M). They carry tenor information and optional reset lags.

We create two SOFR indices and sample a forward rate over a custom accrual window.

In [ ]:
sofr_3m = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [
        (0.0, 0.0450),
        (1.0, 0.0465),
        (3.0, 0.0480),
        (7.0, 0.0490),
    ],
    base_date=as_of,
    reset_lag=2,
    day_count="ACT/360",
)
sofr_6m = ForwardCurve(
    "USD-SOFR-6M",
    0.5,
    [
        (0.0, 0.0455),
        (1.0, 0.0470),
        (3.0, 0.0488),
        (7.0, 0.0505),
    ],
    base_date=as_of,
)

start = 1.0
end = 1.5
print(f"SOFR 3M f({start},{end}) = {sofr_3m.forward(start, end):.4%}")
print(f"SOFR 6M f({start},{end}) = {sofr_6m.forward(start, end):.4%}")


## 3. Credit Hazard & Inflation Curves
Hazard curves describe default intensities and survival probabilities. Inflation curves carry CPI projections for inflation-linked payoffs.

We also show how base-correlation data is stored for structured credit engines.

In [ ]:
hazard_curve = HazardCurve(
    "ACME-HZD",
    as_of,
    [
        (0.0, 0.0120),
        (3.0, 0.0180),
        (5.0, 0.0210),
        (7.0, 0.0230),
    ],
    recovery_rate=0.40,
    issuer="ACME",
    seniority="senior",
)
inflation_curve = InflationCurve(
    "US-CPI",
    base_cpi=305.0,
    knots=[
        (0.0, 305.0),
        (1.0, 308.0),
        (3.0, 316.5),
        (5.0, 330.0),
    ],
)
base_corr = BaseCorrelationCurve(
    "CDX-IG-BC",
    [
        (0.03, 0.10),
        (0.07, 0.14),
        (0.10, 0.18),
        (0.30, 0.22),
        (1.00, 0.25),
    ],
)

print("ACME survival 5y:", hazard_curve.survival(5.0))
print("Default probability 0-5y:", hazard_curve.default_prob(0.0, 5.0))
print("Projected CPI level in 4y:", inflation_curve.cpi(4.0))
print("Inflation rate 0-5y:", inflation_curve.inflation_rate(0.0, 5.0))
print("Base correlation grid:", base_corr.points[:3], "...")


## 4. Volatility Surfaces, FX, and Scalars
Surface objects store implied vol grids used by option pricers. `FxMatrix` keeps bilateral FX quotes with arbitrage checks. Scalar prices capture observables such as spot levels or dividend yields.

In [ ]:
fx_matrix = FxMatrix()
fx_matrix.set_quote(EUR, USD, 1.0850)
fx_matrix.set_quote(USD, JPY, 148.25)
fx_matrix.set_quote(EUR, JPY, 160.90)

cap_surface = VolSurface(
    "USD-CAP-VOL",
    expiries=[0.5, 1.0, 2.0, 5.0],
    strikes=[0.01, 0.02, 0.03, 0.04],
    grid=[
        [0.38, 0.36, 0.34, 0.32],
        [0.35, 0.33, 0.31, 0.30],
        [0.32, 0.30, 0.29, 0.28],
        [0.28, 0.27, 0.26, 0.25],
    ],
)
eq_vol_surface = VolSurface(
    "EQUITY-VOL",
    expiries=[0.25, 0.5, 1.0, 2.0],
    strikes=[120.0, 140.0, 160.0, 180.0],
    grid=[
        [0.28, 0.26, 0.25, 0.24],
        [0.27, 0.25, 0.24, 0.23],
        [0.26, 0.24, 0.23, 0.22],
        [0.25, 0.23, 0.22, 0.21],
    ],
)

spot_scalar = MarketScalar.price(Money(150.0, USD))
div_yield = MarketScalar.unitless(0.015)

eurusd_quote = fx_matrix.rate(EUR, USD, as_of)
print(f"EUR/USD spot={eurusd_quote.rate:.4f} triangulated? {eurusd_quote.triangulated}")
print("Cap vol surface expiries:", cap_surface.expiries)
print("Equity ATM vol (1y, strike 150):", eq_vol_surface.value(1.0, 150.0))
print("Spot scalar:", spot_scalar.value, spot_scalar.currency)
print("Dividend yield:", div_yield.value)


## 5. Assemble a MarketContext
`MarketContext` stores all deterministic data keyed by ID. Insert the curves, surfaces, FX matrix, and scalar prices, then inspect counts and retrieval APIs.

In [ ]:
market = MarketContext()
market.insert_discount(usd_ois)
market.insert_discount(eur_ois)
market.insert_forward(sofr_3m)
market.insert_forward(sofr_6m)
market.insert_hazard(hazard_curve)
market.insert_inflation(inflation_curve)
market.insert_base_correlation(base_corr)
market.insert_surface(cap_surface)
market.insert_surface(eq_vol_surface)
market.insert_fx(fx_matrix)
market.insert_price("EQUITY-SPOT", spot_scalar)
market.insert_price("EQUITY-DIV", div_yield)

market.map_collateral("USD-CS01", "USD-OIS")

print("Counts by type:", market.count_by_type())
print("Curve IDs:", market.curve_ids())
print("Has FX?", market.has_fx)
print("Total objects:", market.total_objects)
print("USD discount curve retrieved?", market.discount("USD-OIS").id)


## 6. Pricing With the Market Context
Once the context is populated, pass it to the standard registry. Any instrument referencing the curve IDs can now be priced with deterministic and risk metrics.

In [ ]:
registry = create_standard_registry()
corp_bond = Bond.fixed_semiannual(
    "ACME-5Y",
    Money(10_000_000, USD),
    0.045,
    date(2024, 1, 2),
    date(2029, 1, 2),
    "USD-OIS",
)

result = registry.price_with_metrics(
    corp_bond,
    "discounting",
    market,
    ["clean_price", "ytm", "duration_mod", "dv01"],
)
print("Present value:", result.value)
print("Clean price:", result.measures.get("clean_price"))
print("Yield-to-maturity:", result.measures.get("ytm"))
print("DV01:", result.measures.get("dv01"))


## Summary
- **Curve construction** uses deterministic `(time, value)` grids for discount, forward, hazard, and inflation projections.
- **Surfaces, FX, and scalars** round out option and cross-currency inputs.
- `MarketContext` is the single source of truth that the pricer registry consumes — once IDs are consistent, any valuation routine can reuse the same deterministic state.